# YOLO26 Object Detection Training and Evaluation

This notebook documents the YOLO26 Nano workflow: environment verification, dataset preparation, two training experiments, loading the repository's trained checkpoint, optional held-out test evaluation, and a single-image TP/FP/FN example.

**Environment:** Google Colab with a GPU runtime.

## Before running

1. Open the public GitHub repository and copy its clone URL.
2. In a Colab cell, clone the repository (replace the placeholder URL):

   ```python
   !git clone https://github.com/YOUR-USERNAME/YOLO26-Object-Detection.git /content/YOLO26-Object-Detection
   ```
3. Upload the dataset ZIP to Google Drive at `MyDrive/YOLO26_Project/YOLO26_Dataset.zip`. The ZIP must contain `images/`, `labels/`, and `data.yaml`.
4. Select **Runtime → Change runtime type → T4 GPU**.

The dataset is not included in this GitHub repository. The trained checkpoint used for evaluation is included at `models/best.pt`.


## 1. Environment Setup

In [ ]:
%pip install -q ultralytics==8.4.211


In [ ]:
import torch
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)


## 2. Repository and Dataset Paths

The repository is expected at `/content/YOLO26-Object-Detection` after cloning it using the instructions above. Do not point `PROJECT_ROOT` at the older project folder.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/YOLO26-Object-Detection")

required_repo_files = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "data.yaml",
    PROJECT_ROOT / "models" / "best.pt",
]

missing_repo_files = [path for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    missing_text = "\\n".join(str(path) for path in missing_repo_files)
    raise FileNotFoundError(
        "The repository was not found or is incomplete. Clone the GitHub "
        "repository to /content/YOLO26-Object-Detection first. Missing:\\n"
        + missing_text
    )

CHECKPOINT_PATH = PROJECT_ROOT / "models" / "best.pt"
RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", PROJECT_ROOT)
print("Bundled trained checkpoint:", CHECKPOINT_PATH)
print("Training output directory:", RUNS_DIR)


## 3. Google Drive and Dataset Preparation

The dataset archive is stored separately from GitHub because the dataset is not included in the repository.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from zipfile import ZipFile
import shutil

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/YOLO26_Project")
ZIP_PATH = DRIVE_PROJECT_DIR / "YOLO26_Dataset.zip"
DATASET_DIR = Path("/content/YOLO26_Dataset")

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found: {ZIP_PATH}. "
        "Place YOLO26_Dataset.zip in MyDrive/YOLO26_Project."
    )

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        relative_path = Path(member.filename.replace("\\\\", "/"))
        destination = (DATASET_DIR / relative_path).resolve()
        if not destination.is_relative_to(DATASET_DIR.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)

print("Dataset extracted to:", DATASET_DIR)


In [ ]:
import yaml

image_extensions = {".jpg", ".jpeg", ".png"}
for split in ["train", "val", "test"]:
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Missing images or labels directory for {split}.")

    images = [
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    ]
    labels = list(label_dir.glob("*.txt"))
    image_stems = [path.stem for path in images]
    label_stems = [path.stem for path in labels]

    if len(image_stems) != len(set(image_stems)):
        raise ValueError(f"Duplicate image filenames in {split}.")
    if len(label_stems) != len(set(label_stems)):
        raise ValueError(f"Duplicate label filenames in {split}.")

    missing_labels = set(image_stems) - set(label_stems)
    extra_labels = set(label_stems) - set(image_stems)
    print(f"{split.upper()}: {len(images)} images, {len(labels)} labels")

    if missing_labels or extra_labels:
        raise ValueError(
            f"Image-label mismatch in {split}: "
            f"{len(missing_labels)} images without labels; "
            f"{len(extra_labels)} labels without images."
        )

SOURCE_YAML = DATASET_DIR / "data.yaml"
if not SOURCE_YAML.is_file():
    raise FileNotFoundError(f"Dataset YAML not found: {SOURCE_YAML}")

with SOURCE_YAML.open("r", encoding="utf-8-sig") as file:
    dataset_config = yaml.safe_load(file)

required_keys = {"train", "val", "test", "nc", "names"}
if not isinstance(dataset_config, dict) or not required_keys.issubset(dataset_config):
    raise ValueError("data.yaml must contain train, val, test, nc, and names.")

dataset_config["path"] = str(DATASET_DIR)
COLAB_YAML_PATH = DATASET_DIR / "data_colab.yaml"
with COLAB_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, sort_keys=False)

DATA_YAML = str(COLAB_YAML_PATH)

for split in ["train", "val", "test"]:
    split_path = DATASET_DIR / dataset_config[split]
    if not split_path.exists():
        raise FileNotFoundError(f"Missing {split} path: {split_path}")

print("Dataset configuration:", DATA_YAML)
print(yaml.safe_dump(dataset_config, sort_keys=False))


## 4. Training Configuration

Both experiments use YOLO26 Nano, the same dataset, image size, batch size, optimizer, and random seed. The second experiment changes **both** the initial learning rate (from `0.01` to `0.001`) and early-stopping patience (from `15` to `20`). Therefore, any difference between the experiments cannot be attributed to the learning-rate change alone.

Training cells are included for reproducibility. Running them again creates new run folders under the repository's `runs/` directory; it does not overwrite the checkpoint already stored in `models/best.pt`.


In [ ]:
from ultralytics import YOLO

print("Dataset YAML:", DATA_YAML)
print("Training outputs:", RUNS_DIR)


## 5. Experiment 1 — Baseline YOLO26n

Initial learning rate: **0.01**; patience: **15**.

In [ ]:
baseline_model = YOLO("yolo26n.pt")

baseline_results = baseline_model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=15,
    seed=42,
    deterministic=True,
    pretrained=True,
    amp=True,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="baseline_yolo26n",
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)


### Baseline Training History

In [ ]:
import pandas as pd
from IPython.display import display

baseline_dir = RUNS_DIR / "baseline_yolo26n"
baseline_csv = baseline_dir / "results.csv"
if not baseline_csv.is_file():
    raise FileNotFoundError(
        f"Baseline results not found: {baseline_csv}. Run Experiment 1 first."
    )

baseline_df = pd.read_csv(baseline_csv)
baseline_df.columns = baseline_df.columns.str.strip()
print("Epochs recorded:", len(baseline_df))

metric_cols = [c for c in baseline_df.columns if c.startswith("metrics/")]
loss_cols = [
    c for c in baseline_df.columns
    if c.startswith("train/") or c.startswith("val/")
]
display(baseline_df[["epoch"] + loss_cols + metric_cols].round(4))

map_col = "metrics/mAP50-95(B)"
if map_col in baseline_df.columns and baseline_df[map_col].notna().any():
    best_row = baseline_df.loc[baseline_df[map_col].idxmax()]
    print("Best validation mAP50-95 epoch:", int(best_row["epoch"]))
    print("Best validation mAP50-95:", round(float(best_row[map_col]), 4))


## 6. Experiment 2 — Lower Learning Rate

Initial learning rate: **0.001**; patience: **20**. The other principal training settings are retained.

In [ ]:
experiment2_model = YOLO("yolo26n.pt")

experiment2_results = experiment2_model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    optimizer="SGD",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    seed=42,
    deterministic=True,
    pretrained=True,
    amp=True,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="experiment2_yolo26n_lr001",
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)


## 7. Load the Repository's Trained Checkpoint

For the published project and the recorded final results, load the checkpoint already included in the repository: `models/best.pt`.

This deliberately does **not** point to `runs/experiment2_yolo26n_lr001/weights/best.pt`. That `runs/` path is created only if you execute Experiment 2 again in this Colab session.


In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Repository checkpoint not found: {CHECKPOINT_PATH}")

best_model = YOLO(str(CHECKPOINT_PATH))

print("Loaded checkpoint:", CHECKPOINT_PATH)
print("Checkpoint size (MB):", round(CHECKPOINT_PATH.stat().st_size / (1024 * 1024), 2))


## 8. Optional Held-Out Test Evaluation

The repository contains the previously generated evaluation figures in `evaluation/experiment2_test/`. The default setting below does not rerun test evaluation.

Set `RUN_FINAL_TEST_EVALUATION = True` only when you intentionally want to evaluate the bundled checkpoint against the test split again. New output is written to a separate folder so existing figures are preserved.


In [ ]:
RUN_FINAL_TEST_EVALUATION = False

EVALUATION_DIR = PROJECT_ROOT / "evaluation" / "experiment2_test"

if RUN_FINAL_TEST_EVALUATION:
    from datetime import datetime

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_evaluation_dir = PROJECT_ROOT / "evaluation" / f"recomputed_test_{timestamp}"

    test_results = best_model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        plots=True,
        project=str(PROJECT_ROOT / "evaluation"),
        name=new_evaluation_dir.name,
        exist_ok=False,
        verbose=True,
    )

    print("\nRECOMPUTED TEST RESULTS")
    print("=======================")
    print(f"Precision:  {test_results.box.mp:.5f}")
    print(f"Recall:     {test_results.box.mr:.5f}")
    print(f"mAP50:      {test_results.box.map50:.5f}")
    print(f"mAP50-95:   {test_results.box.map:.5f}")
    EVALUATION_DIR = new_evaluation_dir
else:
    print("Using the evaluation artifacts already included in the repository.")
    print("Evaluation directory:", EVALUATION_DIR)


## 9. Review Evaluation Artifacts

In [ ]:
if not EVALUATION_DIR.is_dir():
    raise FileNotFoundError(
        f"Evaluation folder not found: {EVALUATION_DIR}. "
        "Check that the repository's evaluation folder is present, "
        "or enable RUN_FINAL_TEST_EVALUATION above."
    )

evaluation_files = sorted(
    path for path in EVALUATION_DIR.rglob("*") if path.is_file()
)

for path in evaluation_files:
    print(f"{path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size / 1024:.1f} KB)")

print("Total files:", len(evaluation_files))


## 10. Single-Image TP, FP and FN Analysis

This section uses `000009.jpg` when it exists in the test split. Otherwise, it selects the first test image that has a matching, non-empty label file. Predictions are matched to ground-truth boxes using class agreement and an IoU threshold of 0.50. Each ground-truth object can be matched only once.

These are image-level results, not overall test-set metrics.


In [ ]:
from PIL import Image
import math

IMAGE_DIR = DATASET_DIR / "images" / "test"
LABEL_DIR = DATASET_DIR / "labels" / "test"

CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

preferred_image = IMAGE_DIR / "000009.jpg"
if preferred_image.is_file() and (LABEL_DIR / "000009.txt").is_file():
    image_path = preferred_image
else:
    candidates = sorted(
        path for path in IMAGE_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in image_extensions
        and (LABEL_DIR / f"{path.stem}.txt").is_file()
        and (LABEL_DIR / f"{path.stem}.txt").stat().st_size > 0
    )
    if not candidates:
        raise FileNotFoundError("No test image with a non-empty matching label file was found.")
    image_path = candidates[0]

label_path = LABEL_DIR / f"{image_path.stem}.txt"

with Image.open(image_path) as image:
    image_width, image_height = image.size

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0

ground_truth = []
for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
    values = line.split()
    if len(values) != 5:
        raise ValueError(f"Invalid annotation at line {line_number}.")
    try:
        class_value = float(values[0])
        xc, yc, width, height = map(float, values[1:])
    except ValueError as error:
        raise ValueError(f"Non-numeric annotation at line {line_number}.") from error

    if not all(math.isfinite(v) for v in [class_value, xc, yc, width, height]):
        raise ValueError(f"Non-finite annotation at line {line_number}.")
    if not class_value.is_integer():
        raise ValueError(f"Invalid class ID at line {line_number}.")
    class_id = int(class_value)

    if not (
        0 <= xc <= 1 and 0 <= yc <= 1
        and 0 < width <= 1 and 0 < height <= 1
        and xc - width / 2 >= 0 and yc - height / 2 >= 0
        and xc + width / 2 <= 1 and yc + height / 2 <= 1
    ):
        raise ValueError(f"Invalid bounding box at line {line_number}.")

    ground_truth.append((
        class_id,
        [
            (xc - width / 2) * image_width,
            (yc - height / 2) * image_height,
            (xc + width / 2) * image_width,
            (yc + height / 2) * image_height,
        ],
    ))

if not ground_truth:
    raise ValueError(f"No ground-truth objects found in {image_path.name}.")

prediction_result = best_model.predict(
    source=str(image_path),
    conf=CONFIDENCE_THRESHOLD,
    imgsz=640,
    device=0,
    verbose=False,
)[0]

predictions = [
    (int(box.cls.item()), box.xyxy[0].cpu().tolist(), float(box.conf.item()))
    for box in prediction_result.boxes
]

matched_ground_truth = set()
true_positives = 0
false_positives = 0

for predicted_class, predicted_box, confidence in sorted(
    predictions, key=lambda item: item[2], reverse=True
):
    best_iou = 0.0
    best_match_index = None

    for index, (actual_class, actual_box) in enumerate(ground_truth):
        if index in matched_ground_truth or predicted_class != actual_class:
            continue
        iou = calculate_iou(predicted_box, actual_box)
        if iou > best_iou:
            best_iou = iou
            best_match_index = index

    if best_match_index is not None and best_iou >= IOU_THRESHOLD:
        true_positives += 1
        matched_ground_truth.add(best_match_index)
    else:
        false_positives += 1

false_negatives = len(ground_truth) - len(matched_ground_truth)
precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives else 0.0
)
recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives else 0.0
)
f1_score = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

print("=" * 50)
print("SINGLE TEST IMAGE EVALUATION")
print("=" * 50)
print("Image:", image_path.name)
print("Ground-truth objects:", len(ground_truth))
print("Predicted objects:", len(predictions))
print("True Positives (TP):", true_positives)
print("False Positives (FP):", false_positives)
print("False Negatives (FN):", false_negatives)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("IoU matching threshold:", IOU_THRESHOLD)


## 11. Previously Recorded Worked Example

The earlier notebook recorded the following values for `000009.jpg` with the Experiment 2 model, confidence threshold 0.25, and IoU threshold 0.50:

| Measure | Recorded result |
|---|---:|
| Ground-truth objects | 4 |
| Predictions | 5 |
| True Positives (TP) | 4 |
| False Positives (FP) | 1 |
| False Negatives (FN) | 0 |
| Precision | 0.8000 (80.00%) |
| Recall | 1.0000 (100.00%) |
| F1-score | 0.8889 (88.89%) |

These are historical, image-level results for one example, not the overall held-out test-set metrics. The code cell above recomputes the values using the repository checkpoint and current dataset. If the selected image differs from `000009.jpg`, or the recomputed output differs, use the new output rather than treating this historical table as current.

## 12. Recorded Overall Test Metrics

The project report records the following final test-set metrics for Experiment 2:

| Metric | Test result |
|---|---:|
| Precision | 83.791% |
| Recall | 74.109% |
| mAP50 | 83.033% |
| mAP50-95 | 63.317% |
| GPU inference | 3.4 ms/image |

These values are included as the project's recorded results. Running the optional test-evaluation cell above may produce slightly different values depending on the runtime and evaluation conditions.
